<a href="https://colab.research.google.com/github/febinemmanuel/assessment/blob/main/cnn_casestudy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Libraries

In [1]:
import tensorflow as tf
from keras.datasets import cifar100
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.initializers import HeNormal
import matplotlib.pyplot as plt
import numpy as np

#Read Dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar100.load_data()

  7028736/169001437 ━━━━━━━━━━━━━━━━━━━━ 1:00:03 22us/step

##Image Normalization

In [ ]:
print(x_train.shape)
print(x_test.shape)

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

In [ ]:
# Build the Custom CNN
model = Sequential([

    Conv2D(32,(3,3),
           activation='relu',
           kernel_initializer=HeNormal(),
           kernel_regularizer=l2(0.001),
           input_shape=(32,32,3)),
    BatchNormalization(),

    Conv2D(32,(3,3),activation='relu'),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Conv2D(64,(3,3),activation='relu'),
    BatchNormalization(),

    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Flatten(),

    Dense(256,activation='relu'),
    Dropout(0.5),

    Dense(100,activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'custom_cnn.keras',
    save_best_only=True
)

##Train the Model

In [ ]:
history = model.fit(x_train,y_train,epochs=15,batch_size=64,validation_split=0.2,
                    callbacks=[early_stop, checkpoint]
                    )

In [ ]:
# Evaluate the model
loss, accuracy = model.evaluate(x_test, y_test)

print("Test Accuracy:", accuracy)

#VGG16 Model

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(32,32,3)
)

base_model.trainable = False

vgg_model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(100, activation='softmax')
])

vgg_model.summary()

In [ ]:
vgg_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:

history_vgg = vgg_model.fit(
    x_train,
    y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

In [ ]:
# evaluate the model
vgg_loss, vgg_acc = vgg_model.evaluate(x_test, y_test)

print("VGG16 Test Accuracy:", vgg_acc)

In [ ]:
# compare results
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Custom CNN", "VGG16"],
    "Test Accuracy": [accuracy, vgg_acc]
})

comparison

In [ ]:
# plot model architecture
from tensorflow.keras.utils import plot_model

plot_model(model,
           show_shapes=True,
           show_layer_names=True)

plot_model(vgg_model,
           show_shapes=True,
           show_layer_names=True)